<a href="https://colab.research.google.com/github/yang200212/MSSP-6070/blob/main/Modules/Week_9/PARTICIPATION_ACTIVITY_Week_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
from google.colab import userdata
import csv
import requests
import os
import subprocess
from pathlib import Path
from urllib.parse import urlparse

owner = "yang200212" # Replace with the GitHub repository owner
repository = "MSSP-6070" # Replace with the GitHub repository name
repo_url = f'https://github.com/{owner}/{repository}.git'
repo_path = Path('/content') / repository

# Clone the repository
import subprocess
subprocess.run(['git', 'clone', repo_url])

# Navigate into the cloned repository directory (optional, but often useful)
os.chdir(repository)
print("Changed directory to: {os.getcwd()}")

Changed directory to: {os.getcwd()}


In [11]:
folder = repo_path / "Modules" / "Week_9" / "Downloaded_Images"
folder.mkdir(parents=True, exist_ok=True)

print("Images will be saved to:", folder)

Images will be saved to: /content/MSSP-6070/Modules/Week_9/Downloaded_Images


In [12]:
import subprocess

subprocess.run(
    ['git', '-C', str(repo_path), 'pull', 'origin', 'main'],
    check=True
)

CompletedProcess(args=['git', '-C', '/content/MSSP-6070', 'pull', 'origin', 'main'], returncode=0)

In [13]:
# I will download pictures from Wikepedia Commons
category = "Cats"
max_images = 20

# Wikimedia Commons API
url = "https://commons.wikimedia.org/w/api.php"

headers = {
    "User-Agent": "StudentImageDownloader/1.0 (https://github.com/yang200212/MSSP-6070)"
}

params = {
    "action": "query",
    "generator": "categorymembers",
    "gcmtitle": "Category:" + category,
    "gcmtype": "file",
    "gcmlimit": 50,
    "prop": "imageinfo",
    "iiprop": "url|mime",
    "iiurlwidth": 800,
    "format": "json"
}

image_urls = []
continuation = None

while len(image_urls) < max_images:
    query_params = params.copy()

    if continuation:
        query_params["gcmcontinue"] = continuation

    response = requests.get(
        url,
        params=query_params,
        headers=headers,
        timeout=30
    )
    response.raise_for_status()

    data = response.json()
    pages = data.get("query", {}).get("pages", {})

    for page in pages.values():
        if "imageinfo" not in page:
            continue

        info = page["imageinfo"][0]
        mime = info.get("mime", "")

        if mime in ["image/jpeg", "image/png", "image/webp"]:
            image_url = info.get("thumburl", info.get("url"))
            source_url = info.get("descriptionurl", "")

            image_urls.append((image_url, source_url, mime))

            if len(image_urls) >= max_images:
                break

    continuation = data.get("continue", {}).get("gcmcontinue")

    if not continuation:
        break

print("Found", len(image_urls), "images")

Found 20 images


In [14]:
# Download images
downloaded = 0
source_records = []

extensions = {
    "image/jpeg": ".jpg",
    "image/png": ".png",
    "image/webp": ".webp"
}

for image_url, source_url, mime in image_urls:

    try:
        res = requests.get(
            image_url,
            stream=True,
            headers=headers,
            timeout=30
        )
        res.raise_for_status()

        extension = extensions[mime]
        filename = f"image_{downloaded + 1}{extension}"
        filepath = folder / filename

        with open(filepath, "wb") as image_file:
            for chunk in res.iter_content(8192):
                if chunk:
                    image_file.write(chunk)

        downloaded += 1
        source_records.append([filename, source_url])

        print("Downloaded:", filename)

    except requests.RequestException as error:
        print("Download failed:", error)

# Save source URLs
with open(folder / "image_sources.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Filename", "Source URL"])
    writer.writerows(source_records)

print("Total images downloaded:", downloaded)
print("Saved to:", folder)

Downloaded: image_1.jpg
Downloaded: image_2.jpg
Downloaded: image_3.png
Downloaded: image_4.jpg
Downloaded: image_5.jpg
Downloaded: image_6.jpg
Downloaded: image_7.jpg
Downloaded: image_8.jpg
Downloaded: image_9.jpg
Downloaded: image_10.jpg
Downloaded: image_11.jpg
Downloaded: image_12.jpg
Downloaded: image_13.png
Downloaded: image_14.jpg
Downloaded: image_15.jpg
Downloaded: image_16.jpg
Downloaded: image_17.jpg
Downloaded: image_18.jpg
Downloaded: image_19.jpg
Downloaded: image_20.jpg
Total images downloaded: 20
Saved to: /content/MSSP-6070/Modules/Week_9/Downloaded_Images
